# Progressive Cramming — interactive demo

**Compress a span of text into a single learnable *memory embedding* of a frozen language model — and watch it reconstruct.** Hands-on companion to the paper *Progressive Cramming: Reliable Token Compression and What It Reveals* — runs on a free **Colab T4 GPU**.

> ⚠️ **Requires a GPU.** Runtime ▸ Change runtime type ▸ T4 GPU, then run cells top to bottom.

**Sections.** 1 · Setup &middot; 2 · What is cramming? &middot; 3 · Gallery &middot; 4 · Total vs Progressive &middot; 5 · Compress your own &middot; 6 · Advanced

---
## 1 · Setup

Install the package and the small UI dependency, then load the frozen model. The first install pulls `transformers`/`torch` and takes a couple of minutes.

In [ ]:
# Config — edit here if you fork the demo.
MODEL_CHECKPOINT = "unsloth/Llama-3.2-1B"  # ungated Llama-3.2-1B mirror (no HF license gate)
GALLERY_REPO_ID  = "LarryLovestein/progressive_cramming_demo_gallery"  # pre-computed embeddings
MAX_SEQ_LEN      = 64  # interactive cramming cap (keeps the demo snappy on a T4)
REPO_URL         = "https://github.com/FusionBrainLab/progressive_cramming"
SLIDES_URL       = "https://fusionbrainlab.github.io/progressive_cramming/slides/"

In [ ]:
%pip install -q "git+https://github.com/FusionBrainLab/progressive_cramming.git" ipywidgets

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU, then re-run."
)
print("GPU:", torch.cuda.get_device_name(0))

from progressive_cramming.demo import (
    load_frozen_model, cram_text, progressive_cram_text, reconstruct_text,
)

# fp16 runs natively on a T4 (Turing has no hardware bf16); the compression
# embedding itself is always optimised in float32 inside cram_text.
model, tokenizer = load_frozen_model(MODEL_CHECKPOINT, dtype="float16")
print("Loaded", MODEL_CHECKPOINT, "| hidden size:", model.config.hidden_size)

A couple of small display helpers used throughout (token-level coloured diff and an embedding loader).

In [ ]:
from progressive_cramming.demo.notebook import (
    emb_from_row, render_token_diff, reconstruct_and_show,
)

---
## 2 · What is cramming?

**Cramming** freezes a pretrained language model and optimises *one* input embedding — by gradient descent — until the frozen model reconstructs the original tokens from it. Model weights never change.

**Progressive cramming** grows the target span stage by stage and stops at the **compression horizon**: the longest prefix that reconstructs *exactly*. Each new stage **warm-starts** from the previous embedding.

<p align="center">
<img src="https://raw.githubusercontent.com/FusionBrainLab/progressive_cramming/main/page/assets/fig_progressive.png" width="720" alt="Progressive cramming schematic">
</p>

📄 [Paper & code](https://github.com/FusionBrainLab/progressive_cramming) &middot; 🎬 [Slides](https://fusionbrainlab.github.io/progressive_cramming/slides/) &middot; 🗂 [Trajectories dataset](https://huggingface.co/datasets/mrsndmn/progressive_cramming_trajectories)

---
## 3 · Pre-compressed gallery

Five paragraphs from different domains — each crammed into **one memory embedding** *(cached on the Hub so you don't wait for optimisation)*. Click **▶ Reconstruct** to greedy-decode it back. Green = matches the original at that position.

In [ ]:
from datasets import load_dataset
from progressive_cramming.demo.notebook import display_gallery

gallery_ds = load_dataset(GALLERY_REPO_ID, split="train")
gallery_rows = [r for r in gallery_ds if r["kind"] == "gallery"]

display_gallery(gallery_rows, model=model, tokenizer=tokenizer)

---
## 4 · Total vs Progressive cramming

Two side-by-side pairs on the **same passage** (PG19 sample #7), illustrating two TC failure modes:

- **Llama-3.2-1B (128 tokens).** TC's one residual error lands on the very first content token; greedy decoding diverges immediately. PC reconstructs all 128 tokens.
- **SmolLM2-360M (32 tokens).** TC produces the first ~6 tokens correctly, then drifts into a *coherent but different* enumeration ("four things… should not have to worry about"). PC reconstructs all 32 tokens.

Both TC embeddings reach ≥ 0.99 teacher-forced accuracy. The difference is **what greedy decoding does** with that residual error: PC has none, TC has one.

In [ ]:
from progressive_cramming.demo import load_frozen_model
from progressive_cramming.demo.notebook import display_side_by_side

# Side-by-side compares two frozen models on the same passage. Llama-3.2-1B is
# already loaded above (cell 4); load SmolLM2-360M here so the widget itself
# never has to touch the GPU.
smollm_model, smollm_tokenizer = load_frozen_model("HuggingFaceTB/SmolLM2-360M", dtype="float16")

tc_pc_rows = [r for r in gallery_ds if r["kind"] == "tc_pc"]

display_side_by_side(
    tc_pc_rows,
    models={
        MODEL_CHECKPOINT: (model, tokenizer),
        "HuggingFaceTB/SmolLM2-360M": (smollm_model, smollm_tokenizer),
    },
)

---
## 5 · Compress your own text

Type a short text, pick one of the loaded models, hit **▶ Compress**. The widget runs *progressive cramming* live: loss + teacher-forced reconstruction on the left, the embedding's PCA trajectory on the right — points coloured by current stage so the warm-start jumps between stages are visible. The run stops at the **compression horizon** and the final decode appears below with the same green / red diff as §3 and §4.

In [ ]:
from progressive_cramming.demo.notebook import display_interactive_compress

interactive_models = {
    "Llama-3.2-1B": (model, tokenizer),
    "SmolLM2-360M": (smollm_model, smollm_tokenizer),
}

display_interactive_compress(interactive_models, max_seq_len=MAX_SEQ_LEN)

---
## 6 · Advanced

The paper's variants are selected with a few flags on the full repo. Pick options below to assemble the exact command. Run it in a clone of [the repository](https://github.com/FusionBrainLab/progressive_cramming) (a GPU job, heavier than this demo).

- **Activation alignment (hybrid)** — also match the frozen model's hidden states, not just the output tokens.
- **Low-dim projection (K)** — optimise the embedding inside a learned rank-K subspace instead of the full hidden size.

In [ ]:
method_dd  = widgets.Dropdown(options=["full", "progressive", "low-dim"], value="full", description="method")
lowdim_k   = widgets.IntSlider(value=32, min=8, max=256, step=8, description="low-dim K")
align_chk  = widgets.Checkbox(value=False, description="activation alignment (hybrid)")
build_btn  = widgets.Button(description="Build command", button_style="primary")
out6       = widgets.Output()

def build_cmd(_):
    out6.clear_output()
    with out6:
        flags = [
            "python scripts/run_cramming.py",
            f"--model_checkpoint {MODEL_CHECKPOINT}",
            "--dataset_name LarryLovestein/pg19_1k --limit_dataset_items 4",
            f"--max_sequence_length {MAX_SEQ_LEN}",
            "--embedding_init_method random0.02 --learning_rate 0.1",
        ]
        if method_dd.value == "progressive":
            flags.append("--progressive_train 1 --progressive_step 8 --max_optimization_steps_per_token 500")
        if method_dd.value == "low-dim":
            flags.append(f"--low_dim_train --low_dim_size {lowdim_k.value}")
        if align_chk.value:
            flags.append("--loss_type cosine --hybrid_alpha 1.0 --num_alignment_layers 8")
        print(" \\\n    ".join(flags))

build_btn.on_click(build_cmd)
display(widgets.VBox([method_dd, lowdim_k, align_chk, build_btn, out6]))

---
### Citation

```bibtex
@inproceedings{tarasov2026progressive,
  title     = {Progressive Cramming: Reliable Token Compression and What It Reveals},
  author    = {Tarasov, Dmitrii and Lashukov and Goncharova, Elena and Kuznetsov, Andrey},
  year      = {2026},
}
```